# Feature Engineering — AI Engineering Interview Prep

Encoding, scaling, pipelines, feature selection, target encoding.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder,
    OneHotEncoder, OrdinalEncoder, PolynomialFeatures
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel, RFE, mutual_info_classif
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Ready")

In [ ]:
# Create a mixed dataset with numeric and categorical features
n = 500
df = pd.DataFrame({
    'age':      np.random.randint(18, 70, n),
    'income':   np.random.lognormal(10.5, 0.5, n),
    'score':    np.random.normal(65, 15, n).clip(0, 100),
    'gender':   np.random.choice(['M', 'F', 'Other'], n),
    'city':     np.random.choice(['NYC', 'LA', 'Chicago', 'Austin', 'Miami'], n, p=[0.3,0.25,0.2,0.15,0.1]),
    'edu':      np.random.choice(['HS', 'Bachelor', 'Master', 'PhD'], n),
    'target':   np.random.binomial(1, 0.4, n)
})
# Introduce missing
df.loc[np.random.choice(n, 30), 'income'] = np.nan
df.loc[np.random.choice(n, 20), 'city']   = np.nan
print(df.head(3))
print(f"\nMissing: {df.isnull().sum().to_dict()}")

## 1. Scaling

In [ ]:
data = np.array([5, 10, 15, 100, 200, 1000, 1]).reshape(-1, 1)

scalers = {
    'Original':       data,
    'StandardScaler': StandardScaler().fit_transform(data),
    'MinMaxScaler':   MinMaxScaler().fit_transform(data),
    'RobustScaler':   RobustScaler().fit_transform(data),  # uses median/IQR
}

pd.DataFrame({k: v.ravel().round(3) for k, v in scalers.items()})

## 2. Categorical Encoding

In [ ]:
# One-Hot Encoding
ohe = OneHotEncoder(sparse_output=False, drop='first')
city_encoded = ohe.fit_transform(df[['city']].fillna('Unknown'))
print("OHE categories:", ohe.categories_)
print("Shape:", city_encoded.shape)

# Ordinal Encoding (when order matters)
edu_order = [['HS', 'Bachelor', 'Master', 'PhD']]
oe = OrdinalEncoder(categories=edu_order)
edu_encoded = oe.fit_transform(df[['edu']])
print("\nOrdinal encoding of education:")
print(pd.DataFrame({'edu': df['edu'][:5].values, 'encoded': edu_encoded[:5, 0]}))

In [ ]:
# Target Encoding (mean encoding) — with cross-fold to avoid leakage
from sklearn.model_selection import KFold

def target_encode(df, col, target, n_splits=5):
    """Out-of-fold target encoding to prevent leakage."""
    encoded = np.zeros(len(df))
    global_mean = df[target].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    for train_idx, val_idx in kf.split(df):
        means = df.iloc[train_idx].groupby(col)[target].mean()
        encoded[val_idx] = df.iloc[val_idx][col].map(means).fillna(global_mean)
    return encoded

df['city_te'] = target_encode(df.fillna('Unknown'), 'city', 'target')
print("City target encoding (sample):")
print(df[['city', 'city_te']].drop_duplicates().dropna().sort_values('city_te'))

## 3. Full Pipeline with ColumnTransformer

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

numeric_features = ['age', 'income', 'score']
categorical_features = ['gender', 'city', 'edu']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(n_estimators=100, random_state=42))
])

X = df.drop(columns=['target', 'city_te'])
y = df['target']

scores = cross_val_score(pipeline, X, y, cv=5, scoring='roc_auc')
print(f"CV ROC-AUC: {scores.mean():.4f} ± {scores.std():.4f}")

## 4. Feature Selection

In [ ]:
data = load_breast_cancer()
X_bc, y_bc = data.data, data.target

# Mutual Information
mi = mutual_info_classif(X_bc, y_bc, random_state=42)
mi_df = pd.Series(mi, index=data.feature_names).sort_values(ascending=False)
print("Top 10 features by Mutual Information:")
print(mi_df.head(10).round(3))

# SelectFromModel (tree-based)
selector = SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=42))
selector.fit(X_bc, y_bc)
selected = data.feature_names[selector.get_support()]
print(f"\nSelectFromModel selected {len(selected)} features:")
print(list(selected))

## Interview Tips

- **Use Pipeline**: prevents data leakage — fit_transform on train, transform on test. Never fit on test!
- **StandardScaler** assumes Gaussian; **RobustScaler** better for outliers; **MinMaxScaler** for neural nets.
- **OHE vs Ordinal**: OHE for nominal (no order); Ordinal for ordered categories (Low/Med/High).
- **Target encoding**: powerful but leaks if not done out-of-fold. Use `category_encoders.TargetEncoder`.
- **Feature selection**: MI for non-linear; RFE for model-aware; SelectFromModel for tree models.
- **High cardinality categoricals** (cities, userIDs): use target encoding, hashing, or embeddings.